In [1]:
import glob
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns

import h5py

import os.path as op
import os
from os.path import join, exists
import glob
import shutil
from datetime import datetime
import math

from nilearn import plotting
from nilearn import image
from nilearn import masking
import nilearn

import nibabel as nib

import scipy

import csv

from tqdm import tqdm

In [2]:
datestring = datetime.now()
print(datestring)
timestampStr = datestring.strftime("%b%d_%Y")
print(timestampStr)

2025-04-23 10:07:43.726304
Apr23_2025


In [3]:
##define variables

sub_uid='1212' #CHANGE PER SUB

task_name= 'LBLLM' 

lbllm_model = 'TYPED_FITHRF_GLMDENOISE_RR.hdf5'
exp_num_trials=880 
exp_num_trs = 196

localizer_id = 'langlocSN'
rois = {1:'LH_IFGorb', 2:'LH_IFG', 3:'LH_MFG', 4:'LH_AntTemp', 5:'LH_PostTemp', 6:'LH_AnG', 7:'RH_IFGorb', 8:'RH_IFG', 9:'RH_MFG', 10:'RH_AntTemp', 11:'RH_PostTemp', 12:'RH_AnG', 13: 'ALL'}

In [4]:
##set paths 

# set path to stimsets and stimset name
path_to_stimsets = '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/stimset_outputs/lbllm'
stimset_name = f'stimset_lbllm_{sub_uid}_all_sessions.csv'

#path to design matrices
path_to_design_matrices= '/nese/mit/group/evlab/u/holson/EXPT_LBLLM/analyses/glmsingle/design_matrices/lbllm'

#path to fROI masks
path_to_fROI_mask= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_localizers/langloc/top10percent/{sub_uid}_*_PL2017'

#path to glmsingle output
path_to_glmsingle_output= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/output_glmsingle/output_glmsingle_preproc-swr_pcstop5_fracs-0.05_UID-{sub_uid}'

#path to outputs
path_to_outputs= f'/nese/mit/group/evlab/u/holson/LBLLM_analysis/output_glmsingle/outputs_{timestampStr}/{sub_uid}'

In [5]:
## get order of itemIDs 
design_matrix_name = f"design_matrices_lbllm_{str(sub_uid)}_all_sessions.pkl"

# open the design matrix - list of np arrays, one for each run 
design_matrix = pd.read_pickle(join(path_to_design_matrices, design_matrix_name))

# dictionary where each run is a key (by run_idx) and each value is an ordered list of the conditions in the run (by item_id)
item_id_run_dict = {}
# list of all item_ids in chronological order (should match glm_single output)
all_item_ids_in_order = []

for i, run_design_matrix in enumerate(design_matrix):
    run_idx = i + 1 
    
    item_id_run_dict[str(run_idx)] = []

    # get shape of design matrix and check that it matches what we expect  
    num_trs, num_conds = run_design_matrix.shape
    assert num_trs == exp_num_trs, "unexpected number of time points in design matrix"
    assert num_conds == exp_num_trials, "unexpected number of conditions in design matrix"

    # for each tr in the design matrix, record the item id of the corresponding condition if there is a 1
    for i, design_row in enumerate(run_design_matrix):
        if np.sum(design_row) != 0:
            item_id = str(np.argwhere(design_row != 0)[0, 0] + 1)
            item_id_run_dict[str(run_idx)].append(item_id)
            all_item_ids_in_order.append(item_id)

print(all_item_ids_in_order)

['121', '361', '101', '21', '321', '661', '501', '201', '1', '61', '241', '181', '801', '221', '621', '521', '341', '741', '401', '461', '41', '381', '261', '861', '821', '141', '601', '301', '701', '581', '681', '761', '161', '441', '721', '781', '481', '81', '281', '561', '541', '641', '841', '421', '243', '823', '423', '583', '623', '863', '323', '483', '303', '543', '165', '3', '443', '763', '263', '43', '663', '83', '783', '463', '723', '103', '703', '643', '383', '23', '363', '403', '283', '123', '343', '503', '563', '203', '63', '683', '603', '183', '523', '743', '223', '803', '143', '843', '125', '645', '465', '605', '765', '145', '405', '365', '245', '325', '665', '205', '285', '265', '825', '625', '305', '485', '345', '65', '445', '225', '845', '545', '745', '25', '85', '185', '5', '45', '565', '685', '705', '505', '525', '805', '105', '725', '385', '585', '785', '425', '163', '865', '347', '627', '327', '207', '107', '447', '387', '187', '247', '707', '787', '47', '647', '67

In [6]:
##get trial responses in language parcels

#load betas across all trials
lbllm_model_file=h5py.File(op.join(path_to_glmsingle_output,lbllm_model),'r')
lbllm_betas_allTrials_dataset=lbllm_model_file['betasmd']

#check dimensions 
print(lbllm_betas_allTrials_dataset)

(x_dim, y_dim, z_dim, num_trials) = lbllm_betas_allTrials_dataset.shape

assert num_trials==exp_num_trials

#convert to numpy array
lbllmm_betas_allTrials=np.zeros((x_dim, y_dim, z_dim, num_trials))
lbllm_betas_allTrials_dataset.read_direct(lbllmm_betas_allTrials)

#create dictionary with raw betas for each fROI and item_id
raw_beta_means = {}

#for each trial...
for trial_idx, item_id in enumerate(all_item_ids_in_order):
    
    raw_beta_means[item_id] = {}
    
    # load trial beta map
    trial_beta_array_img = lbllmm_betas_allTrials[:,:,:,trial_idx]
       
    #for each fROI...    
    for roi_num, roi_label in rois.items():      
               
        # load fROI mask
        froi_mask_name = f'{localizer_id}_{roi_label}_fROI_mask_binary_top10percent.npy'
        froi_mask_img = np.load(glob.glob(join(path_to_fROI_mask,froi_mask_name))[0])
        
        # initialize output
        #output_array_item = np.zeros((x_dim, y_dim, z_dim))
        output_array_item = np.full((x_dim, y_dim, z_dim), np.nan)
        
        # find where mask is 1
        indices = np.where(froi_mask_img == 1)
        
        # get the corresponding values in trial beta map
        corresponding_values = trial_beta_array_img[indices]
        
        # assign the corresponding values to the output array at the specified indices
        output_array_item[indices] = corresponding_values
        
        # saving individual trial arrays        
        export_path = f'{path_to_outputs}/raw_betas/{roi_label}'
        export_filename = f'{export_path}/{task_name}_itemID_{item_id}_{roi_label}_raw_betas'  
        # if no dir make dir
        if not exists(export_path):
            os.makedirs(export_path)
        # saving np array to .npy file
        np.save(f'{export_filename}.npy', output_array_item)
        
        #save non-normalized average betas per fROI        
        raw_beta_means[item_id][roi_label] = np.nanmean(output_array_item)
        
        print('raw beta mean example', raw_beta_means[item_id][roi_label])
        


<HDF5 dataset "betasmd": shape (91, 109, 91, 880), type "<f4">
raw beta mean example 0.362608884687762
raw beta mean example 0.08655252605676651
raw beta mean example -0.19387599227434776
raw beta mean example 0.5237559173884658
raw beta mean example 0.6172437652440395
raw beta mean example 0.3623379500152973
raw beta mean example 0.20620085691680778
raw beta mean example 0.1987132967201372
raw beta mean example -0.90436696815998
raw beta mean example 0.6296141132758073
raw beta mean example 0.6437125203499602
raw beta mean example -0.2256768937007739
raw beta mean example 0.4103283474458753
raw beta mean example 0.23105388044102773
raw beta mean example 0.2995382145807768
raw beta mean example -0.11293839713796339
raw beta mean example 0.5565562022173566
raw beta mean example 0.6500888283913008
raw beta mean example 0.07560844176824992
raw beta mean example 0.3372025837668696
raw beta mean example 0.25956648170948027
raw beta mean example -0.4670227508833434
raw beta mean example 0.36

raw beta mean example 0.04840502932673446
raw beta mean example 0.24077870531710086
raw beta mean example 0.02097542579061925
raw beta mean example -0.6986642000642983
raw beta mean example -0.31745490156114103
raw beta mean example 0.3056658440606391
raw beta mean example -0.20947088103025774
raw beta mean example 0.1948912856575512
raw beta mean example -0.182693573501176
raw beta mean example 0.03729200657750115
raw beta mean example -0.5755241263557125
raw beta mean example -0.13967586612096056
raw beta mean example -0.13571620329302994
raw beta mean example -0.2619517777240807
raw beta mean example -0.1412311380223955
raw beta mean example -0.3160461317891112
raw beta mean example -0.9919696675764548
raw beta mean example -0.15896197039013107
raw beta mean example 0.5849576601164138
raw beta mean example -0.47968833716700504
raw beta mean example -0.10754003396052551
raw beta mean example 0.06866709153669384
raw beta mean example -0.19767978865230118
raw beta mean example 0.695906

raw beta mean example 0.5347258860487102
raw beta mean example -1.273861816606006
raw beta mean example -1.2136443002025286
raw beta mean example -0.8005070775113208
raw beta mean example -0.615354195797096
raw beta mean example -1.0262141313025002
raw beta mean example -0.16776526469397002
raw beta mean example -0.02945850741722294
raw beta mean example -0.29489504256440946
raw beta mean example -1.2304605695795505
raw beta mean example -0.5063670822333521
raw beta mean example -0.2902095248678036
raw beta mean example -0.39501458815155693
raw beta mean example -0.6338599849221501
raw beta mean example -0.9862083192612674
raw beta mean example -0.6169336664304137
raw beta mean example -0.24306486307227232
raw beta mean example -0.5818556936749149
raw beta mean example -0.7286573535052396
raw beta mean example -0.5410750987438055
raw beta mean example -0.009417767863015871
raw beta mean example -0.3531645275404056
raw beta mean example -0.5405905602222427
raw beta mean example -0.52472

raw beta mean example -1.2830754480463393
raw beta mean example -0.5561805688804644
raw beta mean example -1.0778030805163463
raw beta mean example -0.975650626879472
raw beta mean example -0.8408792567867283
raw beta mean example -0.5284195980181297
raw beta mean example -2.2164351058767195
raw beta mean example -0.6104641116847016
raw beta mean example -0.6156945604325856
raw beta mean example -0.43300065845519736
raw beta mean example -0.8407201598880351
raw beta mean example -0.9066402606061987
raw beta mean example -0.4419783286750317
raw beta mean example -0.6038956760944045
raw beta mean example -0.07177238039346698
raw beta mean example -0.6118967100717443
raw beta mean example -0.12937448723241687
raw beta mean example -0.784807269033548
raw beta mean example -0.5786695180088282
raw beta mean example 0.05357934028019217
raw beta mean example -0.5198065272640963
raw beta mean example -0.5919661696053158
raw beta mean example 0.2682635601896506
raw beta mean example -0.445407724

raw beta mean example 0.29526812722190066
raw beta mean example -0.4706880756326624
raw beta mean example -0.1237724794819951
raw beta mean example 0.4904601417798945
raw beta mean example 0.20643893027452442
raw beta mean example 0.1936414176284462
raw beta mean example 0.2126898802852688
raw beta mean example 0.12890536811237246
raw beta mean example -1.8015220020268414
raw beta mean example -1.260635795990626
raw beta mean example -0.8812521971920704
raw beta mean example -0.523202001365713
raw beta mean example -1.122066158397218
raw beta mean example -0.252184584358922
raw beta mean example -0.7073113035309959
raw beta mean example -0.5626896410466482
raw beta mean example -1.00792731536909
raw beta mean example -0.7333999821756241
raw beta mean example -0.4414110395268111
raw beta mean example -0.39539903736306137
raw beta mean example -0.752575153438831
raw beta mean example -2.4355851572913094
raw beta mean example -1.2859984358151755
raw beta mean example -1.027691719379831
ra

...

raw beta mean example -0.32406670399049814
raw beta mean example -0.14913595222690904
raw beta mean example -0.14315214098541665
raw beta mean example -0.07754663145706918
raw beta mean example 0.16571506340610417
raw beta mean example 0.3781925963610411
raw beta mean example -0.8419427928475148
raw beta mean example 0.5778613337587016
raw beta mean example -0.03797588100562156
raw beta mean example -0.33002216562342185
raw beta mean example -0.01752082513663385
raw beta mean example 0.44627108822601874
raw beta mean example -0.5162637252733111
raw beta mean example -0.5207702542080525
raw beta mean example -0.46196733552825414
raw beta mean example -0.061876236011226805
raw beta mean example -0.7055873955671604
raw beta mean example -0.9837232352914037
raw beta mean example -0.8008629442254702
raw beta mean example -2.301032860228356
raw beta mean example -0.40729165089779473
raw beta mean example -0.3099739248280303
raw beta mean example -0.5155989551164496
raw beta mean example -0.4

raw beta mean example 0.24135608205924164
raw beta mean example 0.11240856905933469
raw beta mean example 2.53510368250786
raw beta mean example 0.31591720642533405
raw beta mean example 0.6312189250410992
raw beta mean example 1.0674097162743028
raw beta mean example 0.5429290752038407
raw beta mean example 1.4551054239273071
raw beta mean example 0.4555813412057857
raw beta mean example 0.7541263390411722
raw beta mean example 0.6716228278948029
raw beta mean example 1.458001031878136
raw beta mean example 1.1984679015783164
raw beta mean example 0.7898013382344633
raw beta mean example 0.535714469961822
raw beta mean example 3.2096430671975966
raw beta mean example 1.2248930631439698
raw beta mean example 1.2220487025961027
raw beta mean example 1.6219883070542263
raw beta mean example 1.1926556551327814
raw beta mean example 0.21687773073947914
raw beta mean example -0.22023377457944054
raw beta mean example -0.03494067314239417
raw beta mean example -0.08231712674344975
raw beta m

raw beta mean example -0.25387253644232183
raw beta mean example -0.5805505388987714
raw beta mean example -0.901530112128831
raw beta mean example 0.14546565618366003
raw beta mean example -1.1651345052429147
raw beta mean example 0.08443436653275664
raw beta mean example -1.326050109447951
raw beta mean example -1.0830332220537713
raw beta mean example -0.08159727010544453
raw beta mean example 0.43570597177514664
raw beta mean example -0.5566753950380959
raw beta mean example 0.8186237285467418
raw beta mean example 0.5240552419796586
raw beta mean example 0.3057773372832131
raw beta mean example 0.29111562436303706
raw beta mean example 0.138145227797361
raw beta mean example 0.7013543126674798
raw beta mean example -0.6528658724918559
raw beta mean example 0.3025535574990014
raw beta mean example -0.5555777248549969
raw beta mean example -0.9275011331517181
raw beta mean example -0.2590471899452603
raw beta mean example 1.078482430643187
raw beta mean example 0.023959350416469564


raw beta mean example -0.09126523025662074
raw beta mean example -0.31289141320057806
raw beta mean example -0.09976454633998416
raw beta mean example -0.23128900290108645
raw beta mean example -0.10861188874057334
raw beta mean example -0.6430754502983512
raw beta mean example -0.3297324916223685
raw beta mean example -0.07129992298940395
raw beta mean example -0.5835329280206305
raw beta mean example -0.47527867669429064
raw beta mean example -1.1775076994529137
raw beta mean example -0.8069568859362924
raw beta mean example -0.5428020005983611
raw beta mean example -1.693444604569293
raw beta mean example -0.6045454829314186
raw beta mean example -0.5310060211844048
raw beta mean example -0.521931005595252
raw beta mean example -0.5887159522287575
raw beta mean example -0.19874957445505503
raw beta mean example -0.28953525638363015
raw beta mean example -0.18177207674276322
raw beta mean example -0.381280169847987
raw beta mean example -0.10927951174425118
raw beta mean example -0.5

raw beta mean example -0.2213924478466674
raw beta mean example -0.37564946447530667
raw beta mean example -0.9314209384700427
raw beta mean example 0.0019486255437680049
raw beta mean example -0.12162344668664642
raw beta mean example -0.3945253523532301
raw beta mean example -0.7267666072287458
raw beta mean example 0.12837285422462902
raw beta mean example 0.03832460296916444
raw beta mean example 0.2756815173269178
raw beta mean example -0.8830251975639446
raw beta mean example -0.5967311594138542
raw beta mean example -0.3547420070724601
raw beta mean example -0.3368156509233947
raw beta mean example -0.45586942478272496
raw beta mean example -0.26926598336834173
raw beta mean example -0.23393110827093225
raw beta mean example -1.12104651573542
raw beta mean example -0.6573576530814171
raw beta mean example -0.5735594978991975
raw beta mean example -0.26714959747637934
raw beta mean example -0.41846213456779974
raw beta mean example 0.11746539387565393
raw beta mean example -0.940

In [7]:
#get session IDs for each item

#consider updating code to make sure that item order is not dependent on spreadsheet order

# initialize dictionary to map session ids to a map of the corresponding item ids in the order they occurred in 
session_items_dict = {}
session_items_dict["1"] = []
session_items_dict["2"] = []

# get the item_id and corresponding session_id from the stimset 
with open(join(path_to_stimsets, stimset_name), mode = 'r', encoding='utf-8-sig') as file:
    stimset_table = csv.DictReader(file, dialect = 'excel')     # may need to change dialect as needed

    for row in stimset_table:
        item_id = str(int(float(row['item_id'])))
        session_id = row['session_id']

        session_items_dict[session_id].append(item_id)

# get length of each session list and print it
num_items_session1 = len(session_items_dict["1"])
num_items_session2 = len(session_items_dict["2"])
print(f"There are {num_items_session1} items in session 1 and {num_items_session2} items in session 2")

print(session_items_dict["1"])
print(session_items_dict["2"])

# get overall list of items in order and assert that it's the same as the ordered list from the design matrix
all_item_ids_in_order_stimset = session_items_dict["1"] + session_items_dict["2"]
assert all_item_ids_in_order_stimset == all_item_ids_in_order, "the order of item ids from the design matrix does not match the order of item ids from the stimset"


There are 440 items in session 1 and 440 items in session 2
['121', '361', '101', '21', '321', '661', '501', '201', '1', '61', '241', '181', '801', '221', '621', '521', '341', '741', '401', '461', '41', '381', '261', '861', '821', '141', '601', '301', '701', '581', '681', '761', '161', '441', '721', '781', '481', '81', '281', '561', '541', '641', '841', '421', '243', '823', '423', '583', '623', '863', '323', '483', '303', '543', '165', '3', '443', '763', '263', '43', '663', '83', '783', '463', '723', '103', '703', '643', '383', '23', '363', '403', '283', '123', '343', '503', '563', '203', '63', '683', '603', '183', '523', '743', '223', '803', '143', '843', '125', '645', '465', '605', '765', '145', '405', '365', '245', '325', '665', '205', '285', '265', '825', '625', '305', '485', '345', '65', '445', '225', '845', '545', '745', '25', '85', '185', '5', '45', '565', '685', '705', '505', '525', '805', '105', '725', '385', '585', '785', '425', '163', '865', '347', '627', '327', '207', '107'

In [8]:
#z score betas by session

#initialize output csv
output_fname = f'{task_name}_responsesByItem_sub-{sub_uid}_{timestampStr}.csv'
df_betas_perItem = op.join(path_to_outputs, output_fname)


#initialize
z_scored_array_mean_list = []


for session_num in list(session_items_dict.keys()):
    for roi_num in rois:
        roi = rois[roi_num]
        print(roi)
        items_in_session = session_items_dict[session_num]
        #items_in_session_str = [str(item) for item in items_in_session]
        
        print('items in session',items_in_session)
        
        #path to raw beta maps
        path_to_raw_beta_maps = f'{path_to_outputs}/raw_betas/{roi}'
        
        # list to store files matching each session
        filtered_files = []
        for itemid in items_in_session:
            filtered_files.append(f'{task_name}_itemID_{itemid}_{roi}_raw_betas.npy')
            print('item id', itemid)
            
        #print(filtered_files)
        
        # List to store the loaded numpy arrays
        unstacked_raw_arrays = []

        # Load each filtered .npy file and store the array
        for file in tqdm(filtered_files, desc="Loading files"):
            file_path = join(path_to_raw_beta_maps, file)
            array = np.load(file_path)
            unstacked_raw_arrays.append(array)
            
        # stack them along a new axis to create a 4D array
        stacked_raw_array = np.stack(unstacked_raw_arrays, axis=0)
        
        
        #print('stacked array elements',np.unique(stacked_raw_array.reshape(-1)))
    

        print(stacked_raw_array.shape)  # Output will depend on the number and shape of the loaded arrays
        
        # Identify non-NaN voxels across trials (axis=0)
        not_all_nan_voxels = ~np.isnan(stacked_raw_array).all(axis=0)

        # Filter out voxels that are all NaN
        filtered_array = stacked_raw_array[:, not_all_nan_voxels]

        # Check for voxels with zero variance across trials
        zero_variance_voxels = np.std(filtered_array, axis=0) == 0
        
        zero_variance_count = np.sum(zero_variance_voxels)
        print(f"Number of voxels with zero variance: {zero_variance_count}")


        # Apply z-score normalization to the filtered array (excluding zero variance voxels)
        z_score_filtered_array = scipy.stats.zscore(filtered_array[:, ~zero_variance_voxels], axis=0)

        # Create an array of NaNs with the same shape as the original array
        z_score_array = np.full(stacked_raw_array.shape, np.nan)

        # Initialize a mask for non-NaN, non-zero variance voxels
        final_mask = np.full(stacked_raw_array.shape[1:], False)
        final_mask[not_all_nan_voxels] = ~zero_variance_voxels

        # Fill the z-score array with the computed z-scores for non-NaN, non-zero variance voxels
        z_score_array[:, final_mask] = z_score_filtered_array

        # For zero variance voxels, set z-scores to 0
        # Adjusted to handle the boolean indexing correctly
        zero_variance_final_mask = np.full(stacked_raw_array.shape[1:], False)
        zero_variance_final_mask[not_all_nan_voxels] = zero_variance_voxels
        z_score_array[:, zero_variance_final_mask] = 0

        
        
        # unstack it
        unstacked_z_scored_array = np.split(z_score_array, z_score_array.shape[0], axis=0)
        assert len(unstacked_z_scored_array) == len(items_in_session)

        
        for idx, itemid in enumerate(items_in_session):
            
            z_scored_array = unstacked_z_scored_array[idx]
            #print('z scored array elements',np.unique(z_scored_array.reshape(-1)))
            
            export_path = f'{path_to_outputs}/zscored_betas/{roi}'
            export_filename = f'{export_path}/{task_name}_itemID_{itemid}_fROI_zscored_betas.npy'

            if not exists(export_path):
                os.makedirs(export_path)         
            
            # save the z scored map
            np.save(export_filename, z_scored_array)
            
            z_scored_array_mean = np.nanmean(z_scored_array)
            z_scored_array_mean_list.append(z_scored_array_mean)
            
            #print('z score mean',"{:.10f}".format(z_scored_array_mean))
            
            mean_beta_raw = raw_beta_means[itemid][roi]
            #print('raw mean',mean_beta_raw)
            
            
            df_betas_perItem_currentRow = pd.DataFrame({'PartID': sub_uid,
                                                 'localizer': localizer_id,
                                                 'ROI': roi,
                                                 'task': task_name,
                                                 'session': session_num,
                                                 'item_id': itemid,
                                                 'model': lbllm_model,
                                                 'mean_beta_zscored': z_scored_array_mean,
                                                 'mean_beta_raw': mean_beta_raw}, index=[0])
    
            if not os.path.isfile(df_betas_perItem):
                df_betas_perItem_currentRow.to_csv(df_betas_perItem, index=False, header='column_names')
            else: 
                df_betas_perItem_currentRow.to_csv(df_betas_perItem, mode='a', index=False, header=False)



LH_IFGorb
items in session ['121', '361', '101', '21', '321', '661', '501', '201', '1', '61', '241', '181', '801', '221', '621', '521', '341', '741', '401', '461', '41', '381', '261', '861', '821', '141', '601', '301', '701', '581', '681', '761', '161', '441', '721', '781', '481', '81', '281', '561', '541', '641', '841', '421', '243', '823', '423', '583', '623', '863', '323', '483', '303', '543', '165', '3', '443', '763', '263', '43', '663', '83', '783', '463', '723', '103', '703', '643', '383', '23', '363', '403', '283', '123', '343', '503', '563', '203', '63', '683', '603', '183', '523', '743', '223', '803', '143', '843', '125', '645', '465', '605', '765', '145', '405', '365', '245', '325', '665', '205', '285', '265', '825', '625', '305', '485', '345', '65', '445', '225', '845', '545', '745', '25', '85', '185', '5', '45', '565', '685', '705', '505', '525', '805', '105', '725', '385', '585', '785', '425', '163', '865', '347', '627', '327', '207', '107', '447', '387', '187', '247', '70

Loading files: 100%|███████████████████████████████████████████████████████████████████████| 440/440 [00:04<00:00, 98.75it/s]


(440, 91, 109, 91)
Number of voxels with zero variance: 0
LH_IFG
items in session ['121', '361', '101', '21', '321', '661', '501', '201', '1', '61', '241', '181', '801', '221', '621', '521', '341', '741', '401', '461', '41', '381', '261', '861', '821', '141', '601', '301', '701', '581', '681', '761', '161', '441', '721', '781', '481', '81', '281', '561', '541', '641', '841', '421', '243', '823', '423', '583', '623', '863', '323', '483', '303', '543', '165', '3', '443', '763', '263', '43', '663', '83', '783', '463', '723', '103', '703', '643', '383', '23', '363', '403', '283', '123', '343', '503', '563', '203', '63', '683', '603', '183', '523', '743', '223', '803', '143', '843', '125', '645', '465', '605', '765', '145', '405', '365', '245', '325', '665', '205', '285', '265', '825', '625', '305', '485', '345', '65', '445', '225', '845', '545', '745', '25', '85', '185', '5', '45', '565', '685', '705', '505', '525', '805', '105', '725', '385', '585', '785', '425', '163', '865', '347', '627

Loading files: 100%|██████████████████████████████████████████████████████████████████████| 440/440 [00:03<00:00, 130.50it/s]


(440, 91, 109, 91)
Number of voxels with zero variance: 0
LH_MFG
items in session ['121', '361', '101', '21', '321', '661', '501', '201', '1', '61', '241', '181', '801', '221', '621', '521', '341', '741', '401', '461', '41', '381', '261', '861', '821', '141', '601', '301', '701', '581', '681', '761', '161', '441', '721', '781', '481', '81', '281', '561', '541', '641', '841', '421', '243', '823', '423', '583', '623', '863', '323', '483', '303', '543', '165', '3', '443', '763', '263', '43', '663', '83', '783', '463', '723', '103', '703', '643', '383', '23', '363', '403', '283', '123', '343', '503', '563', '203', '63', '683', '603', '183', '523', '743', '223', '803', '143', '843', '125', '645', '465', '605', '765', '145', '405', '365', '245', '325', '665', '205', '285', '265', '825', '625', '305', '485', '345', '65', '445', '225', '845', '545', '745', '25', '85', '185', '5', '45', '565', '685', '705', '505', '525', '805', '105', '725', '385', '585', '785', '425', '163', '865', '347', '627

...

(440, 91, 109, 91)
Number of voxels with zero variance: 0
RH_AnG
items in session ['420', '880', '160', '740', '680', '760', '260', '360', '40', '80', '400', '780', '220', '240', '860', '200', '540', '440', '100', '60', '720', '380', '120', '820', '520', '560', '700', '500', '140', '180', '620', '480', '660', '800', '840', '460', '20', '340', '640', '579', '600', '320', '300', '280', '758', '118', '198', '398', '878', '718', '378', '678', '618', '498', '518', '818', '598', '38', '438', '538', '338', '658', '578', '98', '638', '698', '838', '218', '18', '858', '778', '358', '238', '458', '738', '298', '138', '258', '278', '318', '798', '158', '78', '558', '58', '478', '178', '418', '576', '516', '296', '796', '776', '276', '56', '416', '196', '314', '816', '156', '716', '536', '476', '736', '16', '136', '556', '836', '636', '616', '436', '256', '236', '36', '376', '876', '396', '596', '176', '756', '696', '456', '656', '76', '116', '336', '676', '356', '216', '96', '856', '496', '594', 

Loading files: 100%|██████████████████████████████████████████████████████████████████████| 440/440 [00:03<00:00, 130.28it/s]


(440, 91, 109, 91)
Number of voxels with zero variance: 0
ALL
items in session ['420', '880', '160', '740', '680', '760', '260', '360', '40', '80', '400', '780', '220', '240', '860', '200', '540', '440', '100', '60', '720', '380', '120', '820', '520', '560', '700', '500', '140', '180', '620', '480', '660', '800', '840', '460', '20', '340', '640', '579', '600', '320', '300', '280', '758', '118', '198', '398', '878', '718', '378', '678', '618', '498', '518', '818', '598', '38', '438', '538', '338', '658', '578', '98', '638', '698', '838', '218', '18', '858', '778', '358', '238', '458', '738', '298', '138', '258', '278', '318', '798', '158', '78', '558', '58', '478', '178', '418', '576', '516', '296', '796', '776', '276', '56', '416', '196', '314', '816', '156', '716', '536', '476', '736', '16', '136', '556', '836', '636', '616', '436', '256', '236', '36', '376', '876', '396', '596', '176', '756', '696', '456', '656', '76', '116', '336', '676', '356', '216', '96', '856', '496', '594', '61

Loading files: 100%|██████████████████████████████████████████████████████████████████████| 440/440 [00:03<00:00, 125.56it/s]


(440, 91, 109, 91)
Number of voxels with zero variance: 0
